# 03 — Descriptive statistics của M9–M11 (E6, phục hồi)

**⚠️ TOÀN BỘ SỐ LIỆU BÊN DƯỚI LÀ SYNTHETIC.**

Theo mục **10.8** và **11.2**, E6 chạy 3 lượt, mỗi lượt 5 warmup không tính, cùng manifest kết quả. Báo từng lượt min/median/mean/p95 nearest-rank/max và số FAIL/TIMEOUT. M9 là thời gian tới **kết thúc readback đúng**, không phải `charttime` hay timestamp `stored_at` trước commit.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").exists()), None)
if ROOT is None:
    raise RuntimeError("Notebook must run from this repository folder or one of its subfolders")
SYNTHETIC = ROOT / "data" / "sample_synthetic"
print("Repository:", ROOT)
import matplotlib.pyplot as plt
from analysis_core.timing import (e6_run_summary, nearest_rank, m10_mapping_durations,
                                  m11_recovery_durations, summarize, duration_ms)
e6 = pd.read_csv(SYNTHETIC / "e6_timings.csv")
recovery = pd.read_csv(SYNTHETIC / "recovery_timings.csv")
print("⚠️ SYNTHETIC — DO NOT USE FOR RESEARCH CONCLUSIONS")


## A. E6: M9, thống kê theo run_id

M9 = T1 (kết thúc lần đọc lại đúng) − T0 (LIS commit Publish). Không gán độ trễ 0 cho timeout và không đưa warmup vào thống kê. Với `n` rất nhỏ, p95 không ổn định.

In [ ]:
e6_report = e6_run_summary(e6)
display(e6_report)


## B. Kiểm tra nearest-rank so với pandas mặc định

Với 10 giá trị 1,...,10: nearest-rank p95 = 10 trong khi pandas mặc định dùng nội suy, vì thế hai con số khác nhau.

In [ ]:
example = pd.Series(range(1, 11))
print("Nearest-rank p95:", nearest_rank(example))
print("Pandas default p95:", example.quantile(0.95))


## C. Phân bố M9 theo lượt E6 (chỉ SUCCESS)

Biểu đồ chỉ giúp nhìn hình dạng phân bố, không chứng minh ưu thế của một kiến trúc khi chưa có baseline.

In [ ]:
plots = []
labels = []
for run_id, rows in e6.groupby("run_id", sort=True):
    rows = rows.loc[(~rows["is_warmup"]) & (rows["run_status"] == "VALID") & (rows["outcome"] == "SUCCESS")]
    plots.append(duration_ms(rows["t0_publish_committed"], rows["t1_verified_readback"]).to_numpy())
    labels.append(run_id)
fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot(plots)
ax.set_xticks(range(1, len(labels) + 1))
ax.set_xticklabels(labels)
ax.set(title="SYNTHETIC E6 end-to-end time (successes only)", ylabel="Latency (ms)", xlabel="Run")
plt.show()


## D. M10 — Thời gian ánh xạ dữ liệu

Chỉ đo mốc `mapping_started_at` đến `transformed_at`, không được xem đây là thời gian toàn luồng.

In [ ]:
mapped = e6.loc[(e6["outcome"] == "SUCCESS") & (~e6["is_warmup"])]
print("M10 mapping durations (synthetic, ms):", summarize(m10_mapping_durations(mapped)))


## E. M11 — Thời gian phục hồi

E4/E7/E8 cần mốc recovery rõ nghĩa theo từng kịch bản; số đo tổng từ Publish phải ghi riêng. M12 là **tỷ lệ** phục hồi và được tính trong Notebook 02.

In [ ]:
print("M11 recovery durations (synthetic, ms):", summarize(m11_recovery_durations(recovery)))
print("Reminder: recovery point must be independently verified against recovery_events.jsonl")
